In [ ]:
import sys
import os
import pandas as pd

In [4]:
from services.google_routes_service import *
from services.route_optimizer import *

In [3]:
# Get project root
project_root = os.path.abspath("..")

# Add to Python path
sys.path.append(project_root)

print(project_root)

/home/chaithanaya/Documents/ai-route-optimizer


In [7]:
trips_df = pd.read_csv("../data/historical_trips.csv")

trips_df.head()

,trip_id,driver_id,date,stop_number,location_id,store_name,latitude,longitude,arrival_time,visit_duration_min,distance_to_next_km,travel_time_to_next_min
0,1,D1,2026-04-01,1,L1,Ratnadeep Supermarket,17.455771,78.212505,09:00,14,10.82,16.2
1,1,D1,2026-04-01,2,L25,IKEA Hyderabad,17.360466,78.233094,09:44,37,44.79,67.2
2,1,D1,2026-04-01,3,L40,Apollo Health City,17.588755,78.580981,11:34,18,33.07,49.6
3,1,D1,2026-04-01,4,L31,Cafe Niloufer,17.598930,78.269166,12:48,25,37.58,56.4
4,1,D1,2026-04-01,5,L33,Pista House,17.261137,78.279991,14:21,25,20.01,30.0


In [8]:


# Cache file path
CACHE_FILE = "route_cache.csv"

# Load existing cache if exists
if os.path.exists(CACHE_FILE):

    cache_df = pd.read_csv(CACHE_FILE)

else:

    cache_df = pd.DataFrame(columns=[

        "origin_id",
        "destination_id",

        "origin_store",
        "destination_store",

        "distance_meters",

        "duration_seconds",

        "duration_minutes"
    ])

print("Trips Loaded:", len(trips_df))
print("Cached Routes:", len(cache_df))

Trips Loaded: 2385
Cached Routes: 0


In [9]:
grouped_trips = trips_df.groupby("trip_id")

In [10]:
# Get first 10 unique trip IDs
trip_ids_10 = trips_df["trip_id"].unique()[:10]

# Filter dataframe
trips_df_10 = trips_df[
    trips_df["trip_id"].isin(trip_ids_10)
]

# Group only these 10 trips
grouped_trips_10 = trips_df_10.groupby("trip_id")

In [11]:
grouped_trips_10

In [12]:
def is_route_cached(
    origin_id,
    destination_id
):

    match = cache_df[

        (cache_df["origin_id"] == origin_id) &

        (cache_df["destination_id"] == destination_id)
    ]

    return len(match) > 0

In [31]:
for trip_id, trip_data in grouped_trips_10:
    locations = list(

        zip(
            trip_data["latitude"],
            trip_data["longitude"]
        )
    )
    location_ids = list(
        trip_data["location_id"]
    )
    print(location_ids)

['L1', 'L25', 'L40', 'L31', 'L33', 'L39']
['L48', 'L19', 'L57']
['L32', 'L43', 'L35', 'L53', 'L57']
['L32', 'L21', 'L60', 'L18', 'L11', 'L45', 'L35']
['L5', 'L21', 'L57', 'L25']
['L48', 'L19', 'L57']
['L48', 'L19', 'L57']
['L32', 'L21', 'L60', 'L18', 'L11', 'L45', 'L35']
['L48', 'L19', 'L57']
['L48', 'L19', 'L57']


In [13]:
for trip_id, trip_data in grouped_trips:
    locations = list(

        zip(
            trip_data["latitude"],
            trip_data["longitude"]
        )
    )
    location_ids = list(
        trip_data["location_id"]
    )
    print(location_ids)

['L1', 'L25', 'L40', 'L31', 'L33', 'L39']
['L48', 'L19', 'L57']
['L32', 'L43', 'L35', 'L53', 'L57']
['L32', 'L21', 'L60', 'L18', 'L11', 'L45', 'L35']
['L5', 'L21', 'L57', 'L25']
['L48', 'L19', 'L57']
['L48', 'L19', 'L57']
['L32', 'L21', 'L60', 'L18', 'L11', 'L45', 'L35']
['L48', 'L19', 'L57']
['L48', 'L19', 'L57']
['L5', 'L21', 'L57', 'L25']
['L5', 'L21', 'L57', 'L25']
['L32', 'L43', 'L35', 'L53']
['L32', 'L43', 'L35', 'L53']
['L1', 'L25', 'L40', 'L31', 'L33']
['L32', 'L43', 'L35', 'L53']
['L32', 'L21', 'L60', 'L18', 'L11', 'L45']
['L5', 'L21', 'L57']
['L32', 'L43', 'L35', 'L53']
['L1', 'L25', 'L40', 'L31', 'L33']
['L2', 'L47', 'L40', 'L29', 'L13', 'L38', 'L25']
['L2', 'L47', 'L40', 'L29', 'L13', 'L38', 'L25']
['L32', 'L43', 'L35', 'L53']
['L32', 'L21', 'L60', 'L18', 'L11', 'L45']
['L32', 'L43', 'L35']
['L32', 'L43', 'L35']
['L35', 'L15', 'L7', 'L8', 'L37', 'L3', 'L43']
['L32', 'L21', 'L60', 'L18', 'L11']
['L2', 'L47', 'L40', 'L29', 'L13', 'L38']
['L32', 'L43', 'L35']
['L1', 'L25', 'L4

In [18]:
all_new_rows = []

for trip_id, trip_data in grouped_trips_10:

    print(f"\nProcessing Trip: {trip_id}")

    # -----------------------------------
    # SORT STOPS
    # -----------------------------------

    trip_data = trip_data.sort_values(
        by="stop_number"
    )

    # -----------------------------------
    # EXTRACT LOCATIONS
    # -----------------------------------

    locations = list(

        zip(
            trip_data["latitude"],
            trip_data["longitude"]
        )
    )

    # -----------------------------------
    # STORE IDS/NAMES
    # -----------------------------------

    location_ids = list(
        trip_data["location_id"]
    )

    store_names = list(
        trip_data["store_name"]
    )

    # -----------------------------------
    # CHECK IF ALL ROUTES EXIST
    # -----------------------------------

    missing_routes = False

    n = len(location_ids)

    for i in range(n):

        for j in range(n):

            if i == j:
                continue

            if not is_route_cached(

                location_ids[i],
                location_ids[j]
            ):

                missing_routes = True
                break

    # -----------------------------------
    # SKIP API IF FULLY CACHED
    # -----------------------------------

    if not missing_routes:

        print("Already Cached")
        continue

    # -----------------------------------
    # GOOGLE API CALL
    # -----------------------------------

    print("Calling Google API...")

    response = get_route_matrix(
        locations
    )

    # -----------------------------------
    # STORE MATRIX PAIRS
    # -----------------------------------

    for item in response:

        origin_index = item[
            "originIndex"
        ]

        destination_index = item[
            "destinationIndex"
        ]

        # Skip same-node routes
        if origin_index == destination_index:
            continue

        origin_id = location_ids[
            origin_index
        ]

        destination_id = location_ids[
            destination_index
        ]

        # Skip already cached
        if is_route_cached(
            origin_id,
            destination_id
        ):
            continue

        try:

            distance_meters = item[
                "distanceMeters"
            ]

            duration_seconds = int(
                item["duration"].replace(
                    "s",
                    ""
                )
            )

            duration_minutes = round(
                duration_seconds / 60,
                2
            )

            row = {

                "origin_id":
                    origin_id,

                "destination_id":
                    destination_id,

                "origin_store":
                    store_names[origin_index],

                "destination_store":
                    store_names[destination_index],

                "distance_meters":
                    distance_meters,

                "duration_seconds":
                    duration_seconds,

                "duration_minutes":
                    duration_minutes
            }

            all_new_rows.append(row)

        except Exception as e:

            print(
                f"Failed pair "
                f"{origin_id} → {destination_id}"
            )

            print(e)


Processing Trip: 1
Already Cached

Processing Trip: 2
Already Cached

Processing Trip: 3
Already Cached

Processing Trip: 4
Already Cached

Processing Trip: 5
Already Cached

Processing Trip: 6
Already Cached

Processing Trip: 7
Already Cached

Processing Trip: 8
Already Cached

Processing Trip: 9
Already Cached

Processing Trip: 10
Already Cached


In [19]:
if len(all_new_rows) > 0:

    new_cache_df = pd.DataFrame(
        all_new_rows
    )

    cache_df = pd.concat(

        [cache_df, new_cache_df],

        ignore_index=True
    )
    all_new_rows=[]
    cache_df.to_csv(
        CACHE_FILE,
        index=False
    )

print("\nRoute cache updated!")

print(
    f"Total cached routes: "
    f"{len(cache_df)}"
)


Route cache updated!
Total cached routes: 528
